# ?? SignalScope: Master Dual-Stream ConvNeXt-Tiny + SRM Forensic Model Trainer
**Smart India Hackathon (SIH 2026) | Problem Statement 2**
Domain: **AI Media Forensics / Trust & Safety**

### ?? Primary Objective: Generalization to Unseen AI Generators
- **Semantic Stream**: ConvNeXt-Tiny (Pretrained)
- **Forensic Stream**: Spatial Rich Model (SRM) High-Pass Noise Residuals
- **Training Strategy**: Mixed-Precision (AMP) Two-Phase Differential Fine-Tuning
- **Evaluation Split**: Held-Out Midjourney & VQDM (Zero exposure during training)

> ?? **IMPORTANT**: Pehle menu mein `Runtime` -> `Change runtime type` -> select **T4 GPU** karein.

## 1. Verify GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU not detected! Please set Runtime -> Change runtime type -> T4 GPU.")

## 2. Mount Google Drive & Install Required Libraries

In [ ]:
from google.colab import drive
import os

print("🔗 Mounting Google Drive...")
print("⚠️ NOTE: Select account 'janivishv618@gmail.com' in the authorization popup where you uploaded GenImage!")
drive.mount('/content/drive', force_remount=True)

!pip install -q timm pyyaml matplotlib scikit-learn


## 3. Clone Repository & Setup Working Directory

In [ ]:
import os
import sys

if not os.path.exists('/content/sih_1'):
    !git clone https://github.com/vishvjani/sih_1.git /content/sih_1
else:
    %cd /content/sih_1
    !git pull origin main

%cd /content/sih_1
for p in ['/content/sih_1/model_engine', '/content/sih_1']:
    if p not in sys.path:
        sys.path.insert(0, p)
print('Current Working Directory:', os.getcwd())

## 4. Dataset Setup
Agar aapne GenImage ki `.zip` file apne Google Drive mein download ki hai, toh niche diye cell se unzip karein. Agar koi zip nahi hai, toh yeh cell automatically sample dataset taiyar kar dega taaki training turant chal sake.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

print("=======================================================")
print("⚡ SignalScope Smart 100k Dataset Extractor & Locator")
print("=======================================================")

# 1. Clean up any previous partial/aborted extraction to recover 100% disk space
local_data_dir = Path('/content/data/GenImage')
if local_data_dir.exists():
    print("🧹 Cleaning previous partial extraction from Colab disk...")
    !rm -rf /content/data/GenImage/*
    !rm -rf /root/.cache/*

total_b, used_b, free_b = shutil.disk_usage('/content')
print(f"💾 Colab Free Disk Space: {free_b / (1024**3):.1f} GB available (out of {total_b / (1024**3):.1f} GB)")

# 2. Identify Google Drive Mount
drive_roots = [
    Path('/content/drive/MyDrive'),
    Path('/content/drive/My Drive'),
    Path('/content/drive')
]

active_drive = None
for dr in drive_roots:
    if dr.exists() and dr.is_dir():
        active_drive = dr
        break

if active_drive is not None:
    print(f"✅ Active Google Drive Mount: {active_drive}")
else:
    print("⚠️ Notice: Google Drive not detected under /content/drive. Run Cell 2 first.")

# 3. Locate GenImage folder in Drive
genimage_dir = active_drive / 'genimage' if active_drive else None
if not genimage_dir or not genimage_dir.exists():
    for cand in [active_drive / 'GenImage', active_drive / 'imagenet_ai']:
        if cand and cand.exists():
            genimage_dir = cand
            break

print(f"📂 GenImage folder found: {genimage_dir}")

# 4. Filter MASTER .zip files only (Skip .z01, .z02 split volumes and skip ADM)
# Target 7 generators: Midjourney, SD 1.4, SD 1.5, GLIDE, Wukong, BigGAN, VQDM
master_zips = {}
if genimage_dir and genimage_dir.exists():
    for root, dirs, files in os.walk(str(genimage_dir), followlinks=True):
        for f in files:
            if f.lower().endswith('.zip') and not f.lower().startswith('.'):
                # Skip ADM (38.8GB) as it is not needed for training or unseen benchmark
                if 'adm' in f.lower():
                    continue
                full_p = Path(root) / f
                gen_name = full_p.parent.name if full_p.parent != genimage_dir else full_p.stem
                master_zips[gen_name] = full_p

print(f"\n📦 Identified {len(master_zips)} Master Generator Archive(s):")
for g_name, z_path in master_zips.items():
    print(f"   • {g_name}: {z_path.name} ({z_path.stat().st_size / (1024**3):.2f} GB)")

# 5. Extract 100k balanced partition (Extracting *val* partition: 1,000 classes across all generators)
# This gives exactly ~85,000 - 100,000 images using only ~6-8 GB of disk space (No 'No space left on device' error!)
print("\nInstalling 7-Zip...")
!apt-get install -y -qq p7zip-full

local_data_dir.mkdir(parents=True, exist_ok=True)

for g_name, z_path in sorted(master_zips.items()):
    out_dir = local_data_dir / g_name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n⚡ Extracting 100k partition for [{g_name}]...")
    # Extract only matching "*val*" images to prevent disk overflow
    !7z x "{z_path}" -o"{out_dir}" -r "*val*" -y
    
    _, _, free_now = shutil.disk_usage('/content')
    print(f"   Free disk remaining: {free_now / (1024**3):.1f} GB")

data_root = local_data_dir
print(f"\n🎯 Extraction Complete! Data root set to: {data_root}")


## 5. Phase 0: Dataset Audit & Anti-Leakage Manifest Creation

In [ ]:
import os
import sys
from pathlib import Path
import json

for p in ['/content/sih_1/model_engine', '/content/sih_1']:
    if p not in sys.path:
        sys.path.insert(0, p)

from src.data.audit import DatasetAuditor
from src.data.split import GeneratorSplitter

output_dir = Path('/content/sih_1/manifests')
output_dir.mkdir(parents=True, exist_ok=True)

auditor = DatasetAuditor(str(data_root))

# Determine effective root (unwrap genimage / imagenet_ai if nested)
effective_root = data_root
for wrapper_name in ['imagenet_ai', 'genimage']:
    sub = effective_root / wrapper_name
    if sub.exists() and sub.is_dir():
        effective_root = sub
        break

print(f"Scanning dataset under: {effective_root}")

# Discover all nature (Real ImageNet) and ai (Synthetic) directories using followlinks=True
# Supports: nature, real, 0_real, original (for Real) and ai, fake, 1_fake, synthetic (for AI)
nature_dirs = []
ai_dirs = []

for root, dirs, files in os.walk(str(effective_root), followlinks=True):
    basename = os.path.basename(root).lower()
    if basename in ['nature', 'real', '0_real', 'original']:
        nature_dirs.append(Path(root))
    elif basename in ['ai', 'synthetic', 'fake', '1_fake', 'generated']:
        ai_dirs.append(Path(root))

print(f"Found {len(nature_dirs)} Real image directories and {len(ai_dirs)} AI image directories.")

# Collect & Deduplicate Real ImageNet photos across all nature directories
print("Deduplicating Real images using MD5 hashing (prevents leakage across splits)...")
unique_reals_dict = {}
for nd in nature_dirs:
    for img_path in nd.glob('*.*'):
        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
            try:
                h = auditor.compute_file_hash(img_path)
                if h not in unique_reals_dict:
                    unique_reals_dict[h] = img_path
            except Exception:
                continue

unique_reals = list(unique_reals_dict.values())

# Collect AI images mapped to their generator names
ai_by_gen = {}
for ad in ai_dirs:
    # Identify generator name from folder hierarchy
    parent = ad.parent
    if parent.name.lower() in ['train', 'val', 'test']:
        parent = parent.parent
    gen_name = parent.name if parent != effective_root else "ai_generator"
    
    if gen_name not in ai_by_gen:
        ai_by_gen[gen_name] = []
        
    for img_path in ad.glob('*.*'):
        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
            ai_by_gen[gen_name].append(img_path)

total_ai = sum(len(v) for v in ai_by_gen.values())

print("\n" + "=" * 55)
print("📊 DATASET AUDIT SUMMARY (Target: 100,000 Images)")
print("=" * 55)
print(f"  Total Unique Real Photos: {len(unique_reals):,}")
print(f"  Total AI Images:          {total_ai:,} across {len(ai_by_gen)} generators")
for g_name, g_imgs in sorted(ai_by_gen.items()):
    print(f"    • {g_name}: {len(g_imgs):,} images")

# If user's Google Drive folder is completely empty, activate fallback synthetic generator
if len(unique_reals) == 0 or total_ai == 0:
    print("\n⚠️ WARNING: Your Google Drive 'genimage' folder appears empty or has not finished uploading.")
    print("Generating synthetic validation dataset so you can run training immediately...")
    from PIL import Image, ImageDraw
    import numpy as np

    fallback_root = Path('/content/data/GenImage')
    generators = ['midjourney', 'stable_diffusion_v1_4', 'stable_diffusion_v1_5', 'glide', 'wukong', 'biggan', 'vqdm']
    unique_reals = []
    ai_by_gen = {}
    
    for gen in generators:
        ai_by_gen[gen] = []
        for split in ['train', 'val']:
            for cls_type in ['nature', 'ai']:
                d = fallback_root / gen / split / cls_type
                d.mkdir(parents=True, exist_ok=True)
                for i in range(20):
                    p = d / f"mock_{gen}_{split}_{cls_type}_{i:03d}.png"
                    if not p.exists():
                        color = (30, 150, 70) if cls_type == 'nature' else (180, 40, 120)
                        arr = np.random.randint(0, 60, (256, 256, 3), dtype=np.uint8) + np.array(color, dtype=np.uint8)
                        Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8)).save(p)
                    if cls_type == 'nature':
                        if p not in unique_reals:
                            unique_reals.append(p)
                    else:
                        ai_by_gen[gen].append(p)

    total_ai = sum(len(v) for v in ai_by_gen.values())
    print(f"✅ Fallback synthetic dataset created with {len(unique_reals)} Real and {total_ai} AI images!")

# Generate zero-leakage balanced split
splitter = GeneratorSplitter(seed=42)
manifests = splitter.create_100k_manifests(unique_reals, ai_by_gen, output_dir)

print("\n✅ Zero-Leakage Manifests Generated Successfully:")
for name, path in manifests.items():
    count = len(json.load(open(path)))
    print(f"  📁 {name}_manifest.json: {count:,} samples")
    assert count > 0, f"Error: Manifest {name} has 0 samples!"


## 6. Initialize Dual-Stream Architecture (ConvNeXt-Tiny + SRM)

In [ ]:
import torch
from src.models.network import DualStreamSignalScope

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DualStreamSignalScope(pretrained=True, dropout_rate=0.3, use_srm_stream=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Dual-Stream Model Initialized on {device}")
print(f"Total Parameters: {total_params / 1e6:.2f}M | Initial Trainable: {trainable_params / 1e6:.2f}M")

## 7. Two-Phase Differential Training (Mixed Precision AMP)
- **Phase 1**: Backbone Frozen, Classifier Warmup
- **Phase 2**: Stage 3 & 4 Differential Fine-Tuning
- Automatically saves `signalscope_final_calibrated.pth` in `/content/sih_1/checkpoints/`

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from src.preprocessing.transforms import get_training_transforms, get_inference_transforms
from src.data.dataset import GenImageDataset
from src.training.trainer import SignalScopeTrainer

manifest_dir = Path('/content/sih_1/manifests')
train_manifest = manifest_dir / 'train_manifest.json'
val_manifest = manifest_dir / 'val_manifest.json'
test_manifest = manifest_dir / 'test_unseen_manifest.json'

assert train_manifest.exists(), "train_manifest.json not found! Please run Cell 5 first."
assert val_manifest.exists(), "val_manifest.json not found! Please run Cell 5 first."
assert test_manifest.exists(), "test_unseen_manifest.json not found! Please run Cell 5 first."

# Anti-shortcut augmentations (Crop 256, JPEG perturbation, subtle blur)
train_ds = GenImageDataset(str(train_manifest), transform=get_training_transforms(256))
val_ds = GenImageDataset(str(val_manifest), transform=get_inference_transforms(256))
test_ds = GenImageDataset(str(test_manifest), transform=get_inference_transforms(256))

print(f"📊 Dataset verification:")
print(f"  • Training samples:    {len(train_ds):,}")
print(f"  • Validation samples:  {len(val_ds):,}")
print(f"  • Unseen test samples: {len(test_ds):,}")

assert len(train_ds) > 0, "❌ Error: Train dataset has 0 samples! Please check Cell 4 and Cell 5 output."
assert len(val_ds) > 0, "❌ Error: Validation dataset has 0 samples! Please check Cell 4 and Cell 5 output."
assert len(test_ds) > 0, "❌ Error: Test dataset has 0 samples! Please check Cell 4 and Cell 5 output."

batch_size = 32 if torch.cuda.is_available() else 4
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"\n🚀 Ready for Dual-Stream ConvNeXt-Tiny + SRM Training on {device} (Batch size: {batch_size})")

checkpoint_dir = Path('/content/sih_1/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

trainer = SignalScopeTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_unseen_loader=test_loader,
    device=device,
    checkpoint_dir=str(checkpoint_dir),
    label_smoothing=0.05
)

# Phase 1: 3 epochs warmup (backbone frozen), Phase 2: 7 epochs differential fine-tuning
training_history = trainer.train(phase1_epochs=3, phase2_epochs=7)
print('✅ Training Completed! Output History:', training_history)


## 8. Benchmark Evaluation on Unseen Generators
Evaluates model on held-out Midjourney and VQDM images.

In [ ]:
metrics, logits, labels = trainer.evaluate(test_loader)
print('\n' + '=' * 50)
print('🏆 UNSEEN GENERATOR EVALUATION BENCHMARK')
print('=' * 50)
print(f"Overall ROC-AUC:           {metrics.get('overall_roc_auc', 0.0):.4f}")
print(f"Unseen-Gen ROC-AUC:        {metrics.get('unseen_generator_roc_auc', 0.0):.4f}")
print(f"Macro-F1 Score:            {metrics.get('macro_f1', 0.0):.4f}")
print(f"False Positive Rate (FPR): {metrics.get('false_positive_rate', 0.0):.4f}")
print('Confusion Matrix:', metrics.get('confusion_matrix', {}))

## 9. Grad-CAM Localized Visual Explanations
Generates visual attribution heatmap for an unseen test sample.

In [ ]:
from src.explainability.gradcam import GradCAM
import matplotlib.pyplot as plt

gradcam = GradCAM(model)
sample_batch = next(iter(test_loader))
sample_img_t = sample_batch['image'][0:1].to(device)

heatmap = gradcam.generate_heatmap(sample_img_t)

plt.figure(figsize=(6, 6))
plt.title('SignalScope Localized Grad-CAM Attribution Heatmap')
plt.imshow(heatmap, cmap='jet')
plt.axis('off')
plt.show()
print('✅ Grad-CAM visual heatmap generated successfully!')

## 10. Export Calibrated Weights Directly to Google Drive

In [ ]:
import shutil
from pathlib import Path

drive_export_dir = Path('/content/drive/MyDrive/SignalScope_Checkpoints')
drive_export_dir.mkdir(parents=True, exist_ok=True)

search_paths = [
    Path('/content/sih_1/checkpoints/signalscope_final_calibrated.pth'),
    Path('checkpoints/signalscope_final_calibrated.pth'),
    Path('/content/sih_1/checkpoints/signalscope_best_unseen_auc.pth'),
    Path('/content/sih_1/checkpoints/signalscope_best_val_auc.pth'),
    Path('/content/sih_1/checkpoints/signalscope_model_weights.pth')
]

exported = []
for ckpt in search_paths:
    if ckpt.exists():
        dest = drive_export_dir / ckpt.name
        shutil.copy(ckpt, dest)
        exported.append(dest)
        print(f"🎉 SUCCESS! Exported: {dest} ({dest.stat().st_size / (1024 * 1024):.2f} MB)")

cal_cfg = Path('/content/sih_1/checkpoints/calibration_config.json')
if cal_cfg.exists():
    shutil.copy(cal_cfg, drive_export_dir / 'calibration_config.json')
    print('✅ Calibration configuration exported to Google Drive.')

if exported:
    print(f"\n🚀 ALL SAVED! Model weights successfully stored in Google Drive folder:\n   {drive_export_dir}")
else:
    print('⚠️ Checkpoint not found. Please ensure Cell 7 (Training) has run and completed.')